# Actual NeMo input/output rails

Use this folder and the shared root kernel. These are synthetic fixtures and a local demo responder. Models are disabled in the NeMo configuration; no credentials are loaded. Offline flags disable telemetry and remote model metadata.

In [ ]:
import os
os.environ.update(DO_NOT_TRACK="1", OTEL_SDK_DISABLED="true", HF_HUB_OFFLINE="1", LITELLM_LOCAL_MODEL_COST_MAP="true")
import json
from pathlib import Path
from pprint import pprint
fixtures = json.loads(Path("docs/dataset/inputs.json").read_text())
pprint(fixtures)

## Read the actual configuration and helper

Read config/config.yml and config/rails.co. `tools/nemo_checks.py` builds actual RailsConfig/LLMRails, registers two ordinary async actions, and records their invocations. `models/demo.py` makes a local response. The framework runs the configured actions; these are custom keyword/regex checks, not pretrained safety models.

In [ ]:
from tools.nemo_checks import build_rails, guarded_request, result_fields
from models.demo import demo_response
action_events = []
rails = build_rails(action_events)
pprint({"input_flows": rails.config.rails.input.flows, "output_flows": rails.config.rails.output.flows, "models": rails.config.models, "custom_data": rails.config.custom_data})

## Local baseline without rails

Actually invoke the local demo on the same fixtures. The synthetic output email is exposed. These outputs are local code results and do not demonstrate language-model behavior.

In [ ]:
baseline_inputs = []
baseline = {}
for item in fixtures:
    baseline_inputs.append(item["text"])
    baseline[item["id"]] = demo_response(item["text"])
pprint(baseline)
pprint({"actual_local_calls": len(baseline_inputs), "model_api_calls": 0})

## Run framework input and output checks

User-only check_async runs input rails; blocked inputs skip the local responder. Assistant-only check_async runs output rails. The SDK values are passed, blocked, modified; a blocked result names its rail. Inspect final exposed output as well as status.

In [ ]:
responder_inputs = []
guarded = {}
for item in fixtures:
    guarded[item["id"]] = await guarded_request(rails, item["text"], demo_response, responder_inputs)
for name, result in guarded.items():
    pprint({"id": name, **result})
pprint({"actual_responder_inputs": responder_inputs, "registered_action_events": action_events})

## Direct output check proves modification

The output rail can operate on an explicit response fixture with no answer-model call. Modification has no blocking-rail name; the action log records which configured action ran and its before/after text.

In [ ]:
output_fixture = "Contact output@example.test."
output_check = await rails.check_async([{"role": "assistant", "content": output_fixture}])
pprint({"before": output_fixture, "framework_result": result_fields(output_check), "last_action_event": action_events[-1]})
assert output_check.status.value == "modified"
assert "@" not in output_check.content
assert guarded["unrelated"]["local_responder_calls"] == 0
assert guarded["normal"]["output_result"]["status"] == "modified"

## Change one configuration value, identical inputs

Change only custom_data.include_vacation. The same vacation question passes, but the unrelated benefits-of-poker collision still passes. Framework execution evidence does not establish semantic safety.

In [ ]:
candidate_events = []
candidate_rails = build_rails(candidate_events, include_vacation=True)
candidate_inputs = []
candidate = {}
for item in fixtures:
    candidate[item["id"]] = await guarded_request(candidate_rails, item["text"], demo_response, candidate_inputs)
for item in fixtures:
    name = item["id"]
    pprint({"id": name, "baseline_input_status": guarded[name]["input_result"]["status"], "candidate_input_status": candidate[name]["input_result"]["status"], "candidate_exposed": candidate[name]["exposed"]})
assert guarded["borderline"]["decision"] == "block"
assert candidate["borderline"]["decision"] == "redact"
assert candidate["collision"]["input_result"]["status"] == "passed"
pprint({"candidate_actual_local_calls": len(candidate_inputs), "registered_input_actions": sum(event["action"] == "allow_hr_input" for event in candidate_events), "registered_output_actions": sum(event["action"] == "mask_email_output" for event in candidate_events), "model_api_calls": 0})